# Response Generation Benchmark — Google Colab
**Llama 3 8B vs Mistral 7B on EmpatheticDialogues / DailyDialog**

This notebook runs the EMAH thesis response-generation benchmark on Colab.

There are **two ways to run inference**, pick one in Section 3:

| Mode | How it works | Best for |
|---|---|---|
| **A. Remote Ollama (tunnel)** | Colab sends requests to your local Ollama server (Carme) via a Cloudflare Tunnel URL | Using your existing pulled models, no GPU needed on Colab |
| **B. Local HF on Colab GPU** | Downloads Llama 3 8B / Mistral 7B in 4-bit and runs them on Colab's T4 GPU | No local server available; standalone run |

The rest of the pipeline (dataset loading, prompts, metrics, plots) is
identical to the local `benchmark_runner.py`.

> **Repo:** `https://github.com/mezlet/LLM-Emotion.git` (branch `benchmark`)


## 1. Check Runtime

For Mode B (local HF inference) you need a GPU runtime:
**Runtime → Change runtime type → T4 GPU**.

For Mode A (remote Ollama via tunnel), a CPU runtime is fine.


In [ ]:
import subprocess
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10).stdout)
except FileNotFoundError:
    print("No GPU detected. This is fine for Mode A (remote Ollama).")
    print("For Mode B, go to Runtime -> Change runtime type -> T4 GPU.")


## 2. Clone Repository & Install Dependencies

Clones the `benchmark` branch of the repo and installs the benchmark's
Python dependencies.

> **Always fetches the latest commit.** If `LLM-Emotion/` already exists
> from a previous cell run this session, it's deleted and re-cloned rather
> than reused — otherwise edits you've pushed to GitHub since starting this
> Colab session would be silently ignored.


In [ ]:
import os, shutil
from pathlib import Path

REPO_URL    = "https://github.com/mezlet/LLM-Emotion.git"
REPO_BRANCH = "benchmark"
REPO_SUBDIR = "benchmark/resp_gen_benchmark"

# Anchor at Colab's standard working directory so re-running this cell
# always lands in the same place, regardless of where a previous run's
# os.chdir(PROJECT_ROOT) left us.
COLAB_ROOT = Path("/content") if Path("/content").exists() else Path.cwd()
os.chdir(COLAB_ROOT)

# Always re-clone fresh so pushed fixes are picked up, even if this cell
# was already run earlier in the same Colab session.
if (COLAB_ROOT / "LLM-Emotion").exists():
    shutil.rmtree(COLAB_ROOT / "LLM-Emotion")

!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL}

PROJECT_ROOT = COLAB_ROOT / "LLM-Emotion" / REPO_SUBDIR
assert PROJECT_ROOT.exists(), f"Expected project at {PROJECT_ROOT}, but it doesn't exist"

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())
!git log -1 --format='Cloned commit: %h %s (%ar)'
!ls


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import sys
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm

from config import MODELS, MODEL_KEYS, DATASETS, GENERATION
from data_loader import load_dataset_samples
from prompt_builder import build_full_prompt, SYSTEM_PROMPT
from evaluator import evaluate_batch, aggregate, load_empathy_classifier, SCORE_COLS
from visualize_results import (
    plot_bar_comparison, plot_radar, plot_latency_distribution,
    plot_scatter_bleu_rouge, plot_emotion_heatmap, generate_all,
)
from IPython.display import Image, display

print("Models:", MODEL_KEYS)
print("Datasets:", list(DATASETS.keys()))


## 3. Choose Inference Backend

### Mode A — Remote Ollama via Cloudflare Tunnel (recommended if available)

If your local Ollama server (e.g. on Carme/cordelia) is exposed via a
Cloudflare Tunnel (`*.trycloudflare.com`), paste that URL below. This lets
Colab use your already-pulled `llama3:8b` / `mistral:7b` models without
downloading anything.

Start a tunnel on your machine with:
```bash
cloudflared tunnel --url http://localhost:11434
```
This prints a URL like `https://random-words-1234.trycloudflare.com` — paste
it as `OLLAMA_HOST` below. **Tunnel URLs are ephemeral** — generate a fresh
one each session.

### Mode B — Local HF inference on Colab's GPU

Leave `OLLAMA_HOST = None` to download and run both models directly on
Colab's GPU in 4-bit (via `bitsandbytes`). Requires a GPU runtime and ~10GB
of download per model on first use.

> **`meta-llama/Meta-Llama-3-8B-Instruct` is a gated model.** You must:
> 1. Request access at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
>    (Meta usually approves within minutes, sometimes instantly)
> 2. Create a token at https://huggingface.co/settings/tokens (read access is enough)
> 3. Authenticate in the cell below before loading any models
>
> `mistralai/Mistral-7B-Instruct-v0.3` is ungated and needs no token, but
> logging in is harmless either way.


In [ ]:
# ── EDIT THIS CELL ───────────────────────────────────────────────────────────

# Mode A: paste your Cloudflare Tunnel URL (e.g. "https://xxxx.trycloudflare.com")
# Mode B: set to None to run models locally on Colab's GPU instead
OLLAMA_HOST = None  # e.g. "https://surgeon-bluetooth-poem-manor.trycloudflare.com"

BACKEND = "ollama" if OLLAMA_HOST else "hf"
print(f"Backend: {BACKEND}" + (f" -> {OLLAMA_HOST}" if OLLAMA_HOST else " (local GPU, 4-bit)"))


In [ ]:
# Required for Mode B only (gated Llama 3 model). Skipped automatically for Mode A.
if BACKEND == "hf":
    from huggingface_hub import login
    from getpass import getpass

    # Prefer Colab's Secrets manager (key icon in left sidebar) if you've
    # added a secret named HF_TOKEN; falls back to an interactive prompt.
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

    if not hf_token:
        hf_token = getpass("Paste your HuggingFace access token (https://huggingface.co/settings/tokens): ")

    login(token=hf_token)
    print("Logged in to HuggingFace Hub ✓")
else:
    print("Mode A (remote Ollama) — no HuggingFace login needed, skipping.")


In [ ]:
from model_client import get_client

if BACKEND == "ollama":
    client = get_client("ollama", host=OLLAMA_HOST)
    if not client.health_check():
        raise RuntimeError(
            f"Cannot reach Ollama at {OLLAMA_HOST}.\n"
            "Check the tunnel is running and the URL is current "
            "(tunnel URLs change every session)."
        )
    print("Ollama reachable ✓")
    print("Available models:", client.list_models())
else:
    print("Using local HF backend — models load lazily on first generate() call.")
    client = get_client("hf", device="auto")


### 3b. 4-bit quantization patch (Mode B only)

`model_client.HuggingFaceClient` loads models in fp16 by default, which
needs ~16GB VRAM per 7-8B model — too much for a single Colab T4 (15GB
total, shared across both models). This cell monkey-patches the loader to
use 4-bit quantization via `bitsandbytes` (~5GB per model), so both models
fit comfortably.

**Skip this cell entirely if using Mode A (remote Ollama).**


In [ ]:
if BACKEND == "hf":
    !pip install -q bitsandbytes accelerate

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    _bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
    )

    def _patched_load(self, model_key):
        if model_key in self._loaded:
            return self._loaded[model_key]
        model_id = MODELS[model_key]["hf_tag"]
        print(f"[model_client] Loading {model_id} in 4-bit …")
        tok = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=_bnb_config,
            device_map="auto",
        )
        self._loaded[model_key] = (tok, model)
        return tok, model

    # Patch the instance method
    import types
    client._load = types.MethodType(_patched_load, client)
    print("4-bit quantization patch applied.")


## 4. Benchmark Configuration

In [ ]:
DATASET_NAME = "empathetic_dialogues"   # or "daily_dialog"
SPLIT        = "test"
NUM_SAMPLES  = 20                       # increase for a full run (slower)
MODE         = "zero_shot"              # "zero_shot" or "few_shot"
MAX_TOKENS   = GENERATION["max_tokens"]
SKIP_EMPATHY = False
EMPATHY_DEVICE = "auto"                 # "auto" | "cpu" | "gpu"


## 5. Load Dataset

Downloads on first use (with retries and fallback sources) and caches
preprocessed samples under `data_cache/`. The repo already ships a cached
`empathetic_dialogues_test_200.json`, so the default config loads instantly.


In [ ]:
samples = load_dataset_samples(
    name=DATASET_NAME,
    split=SPLIT,
    num_samples=NUM_SAMPLES,
)

print(f"Loaded {len(samples)} samples")
pd.DataFrame(samples).head()


In [ ]:
emotion_counts = pd.Series([s["emotion"] for s in samples]).value_counts()
ax = emotion_counts.plot(kind="bar", figsize=(10, 4), title="Emotion category distribution")
ax.set_ylabel("Count")
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()


### Inspect a sample prompt

In [ ]:
example = samples[0]
prompt = build_full_prompt(
    emotion=example["emotion"],
    context=example["context"],
    utterance=example["utterance"],
    mode=MODE,
)

print("SYSTEM PROMPT:\n" + SYSTEM_PROMPT)
print()
print("FULL PROMPT:\n" + prompt)
print()
print("GOLD REFERENCE:\n" + example["reference"])


## 6. Run Inference

Generates a response for every sample with each model.

- **Mode A (remote Ollama):** fast — limited by your tunnel's latency.
- **Mode B (local HF, 4-bit):** first run per model is slow (download +
  quantize); generation itself is reasonably fast on a T4.


In [ ]:
generations = {}  # model_key -> list of (hypothesis, latency)

for model_key in MODEL_KEYS:
    label = MODELS[model_key]["label"]
    results = []
    for sample in tqdm(samples, desc=label):
        hyp, lat = client.generate(
            model_key=model_key,
            emotion=sample["emotion"],
            context=sample["context"],
            utterance=sample["utterance"],
            max_tokens=MAX_TOKENS,
        )
        results.append((hyp, lat))
    generations[model_key] = results

print("Done.")


## 7. Spot-Check Generations

In [ ]:
idx = 0  # change to inspect a different sample

print("EMOTION:   ", samples[idx]["emotion"])
print("UTTERANCE: ", samples[idx]["utterance"])
print("REFERENCE: ", samples[idx]["reference"])
print()
for model_key in MODEL_KEYS:
    hyp, lat = generations[model_key][idx]
    print(f"--- {MODELS[model_key]['label']} ({lat:.2f}s) ---")
    print(hyp)
    print()


## 8. Compute Metrics

BLEU, ROUGE-L, BERTScore F1, and Empathy Score (zero-shot NLI via
`facebook/bart-large-mnli`). On Colab's GPU, the empathy classifier loads on
GPU by default (`EMPATHY_DEVICE = "auto"`); if running Mode B with both LLMs
also on GPU, VRAM may be tight — set `EMPATHY_DEVICE = "cpu"` if you hit OOM.


In [ ]:
empathy_clf = None
if not SKIP_EMPATHY:
    empathy_clf = load_empathy_classifier(EMPATHY_DEVICE)


In [ ]:
records = []
for model_key in MODEL_KEYS:
    hyps = [h for h, _ in generations[model_key]]
    lats = [l for _, l in generations[model_key]]
    refs = [s["reference"] for s in samples]

    metrics = evaluate_batch(refs, hyps, lats, empathy_clf)

    for i, sample in enumerate(samples):
        records.append({
            "model": model_key,
            "label": MODELS[model_key]["label"],
            "mode": MODE,
            "conv_id": sample["conv_id"],
            "emotion": sample["emotion"],
            "utterance": sample["utterance"],
            "reference": refs[i],
            "hypothesis": hyps[i],
            **{col: metrics[col][i] for col in SCORE_COLS},
        })

df = pd.DataFrame(records)
df.head()


## 9. Aggregate Results

In [ ]:
summary_rows = []
for model_key, grp in df.groupby("model"):
    row = {"model": model_key, "label": MODELS[model_key]["label"]}
    for col in SCORE_COLS:
        stats = aggregate(grp[col].tolist())
        row[f"{col}_mean"] = stats["mean"]
        row[f"{col}_std"]  = stats["std"]
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary


In [ ]:
emotion_rows = []
for (model_key, emotion), grp in df.groupby(["model", "emotion"]):
    row = {"model": model_key, "emotion": emotion, "n": len(grp)}
    for col in SCORE_COLS:
        row[f"{col}_mean"] = aggregate(grp[col].tolist())["mean"]
    emotion_rows.append(row)

emotion_df = pd.DataFrame(emotion_rows).sort_values(["emotion", "model"]).reset_index(drop=True)
emotion_df.head(10)


## 10. Visualise Results


In [ ]:
fig_dir = Path("_tmp_figures")
fig_dir.mkdir(exist_ok=True)

tmp_results_dir = Path("_tmp_results")
tmp_results_dir.mkdir(exist_ok=True)
for model_key in MODEL_KEYS:
    df[df["model"] == model_key].to_csv(tmp_results_dir / f"raw_{model_key}.csv", index=False)


In [ ]:
plot_bar_comparison(summary, fig_dir)
display(Image(filename=fig_dir / "comparison_bar.png"))


In [ ]:
plot_radar(summary, fig_dir)
display(Image(filename=fig_dir / "radar_profile.png"))


In [ ]:
plot_latency_distribution(tmp_results_dir, fig_dir)
display(Image(filename=fig_dir / "latency_distribution.png"))


In [ ]:
plot_scatter_bleu_rouge(tmp_results_dir, fig_dir)
display(Image(filename=fig_dir / "scatter_bleu_rouge.png"))


In [ ]:
for metric in ["bleu", "rouge_l", "empathy_score"]:
    plot_emotion_heatmap(emotion_df, metric, fig_dir)
    img_path = fig_dir / f"emotion_heatmap_{metric}.png"
    if img_path.exists():
        display(Image(filename=img_path))


## 11. Save Results & Download

Writes results to `results/<dataset>/` and figures to `figures/<dataset>/`
(matching `benchmark_runner.py`'s layout), then zips both for download.


In [ ]:
import json
import shutil

results_dir = Path("results") / DATASET_NAME
figures_dir = Path("figures") / DATASET_NAME
results_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

summary.to_csv(results_dir / "summary.csv", index=False)
emotion_df.to_csv(results_dir / "per_emotion_summary.csv", index=False)
for model_key in MODEL_KEYS:
    df[df["model"] == model_key].to_csv(results_dir / f"raw_{model_key}.csv", index=False)

def _nan_safe(v):
    return None if isinstance(v, float) and np.isnan(v) else v

with open(results_dir / "all_results.json", "w") as f:
    json.dump([{k: _nan_safe(v) for k, v in r.items()} for r in records], f, indent=2)

generate_all(results_dir, figures_dir)

shutil.rmtree(tmp_results_dir, ignore_errors=True)
shutil.rmtree(fig_dir, ignore_errors=True)

print(f"Results: {results_dir.resolve()}")
print(f"Figures: {figures_dir.resolve()}")


In [ ]:
# Zip results + figures for download
shutil.make_archive("benchmark_output", "zip", root_dir=".", base_dir="results")
shutil.make_archive("benchmark_figures", "zip", root_dir=".", base_dir="figures")

from google.colab import files
files.download("benchmark_output.zip")
files.download("benchmark_figures.zip")


## 12. Summary — Winner per Metric

In [ ]:
METRIC_LABELS = {
    "bleu_mean":          "BLEU ↑",
    "rouge_l_mean":       "ROUGE-L ↑",
    "bert_score_f1_mean": "BERTScore F1 ↑",
    "empathy_score_mean": "Empathy Score ↑",
    "latency_s_mean":     "Latency (s) ↓",
}

for col, label in METRIC_LABELS.items():
    if col not in summary.columns or summary[col].isna().all():
        continue
    if "latency" in col:
        winner = summary.loc[summary[col].idxmin()]
    else:
        winner = summary.loc[summary[col].idxmax()]
    print(f"{label:<22}: {winner['label'].upper():<12} ({winner[col]:.4f})")


## Notes

- **Mode A (remote Ollama)**: latency numbers reflect tunnel + network
  round-trip, not raw model inference time — don't use them for hardware
  comparisons, only for relative model-vs-model comparison.
- **Mode B (local HF, 4-bit)**: 4-bit quantization can slightly shift
  generation quality vs the GGUF-quantized models served by Ollama; note
  this in the thesis methodology if comparing across modes.
- For the full benchmark (`NUM_SAMPLES = 200`), Mode A is much faster
  since no model loading/download is needed on Colab.
- To switch datasets, set `DATASET_NAME = "daily_dialog"` and re-run from
  Section 5.
